In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
from PIL import Image

# ==========================================
# 0. 全局配置与保存路径
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Fed_ViT_Final' # 📁 模型专门存放在这里
os.makedirs(SAVE_DIR, exist_ok=True)

# 确保实验可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
set_seed(42)

print(f"🚀 Running on: {DEVICE}")
print(f"📂 Models will be saved to: ./{SAVE_DIR}/")

# ViT 超参数
IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 # CIFAR-10
NUM_CLASSES_FINETUNE = 6  # NEU-DET

# ==========================================
# 1. 模型定义 (Split-ViT + HE适配双向防御)
# ==========================================
class DynamicAct(nn.Module):
    """
    动态激活函数 (融合第四章逻辑)：
    - mode='gelu': 正常训练，保证收敛
    - mode='poly': 模拟密文计算 (f(x) = ax^2 + bx + c)
    """
    def __init__(self):
        super().__init__()
        self.mode = 'gelu' # 初始默认为明文模式
        self.register_buffer('a', torch.tensor(0.17))
        self.register_buffer('b', torch.tensor(0.5))
        self.register_buffer('c', torch.tensor(0.12))

    def forward(self, x): 
        if self.mode == 'gelu':
            return nn.functional.gelu(x)
        return self.a * (x**2) + self.b * x + self.c

class MockHE(nn.Module):
    """
    【新增】前向特征防御：模拟同态加密层 (来自第四章)
    - 在特征 Z 上添加高斯噪声，模拟 CKKS 加密误差
    """
    def __init__(self):
        super().__init__()
        self.noise_std = 0.0 # 初始无噪声

    def forward(self, x):
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            return x + noise
        return x

class PolyBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), DynamicAct(), nn.Linear(dim * 2, dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        # Client 持有前 2 层
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        # 【新增】前向加密接口
        self.he = MockHE()

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        # 输出前经过 HE 层，开启后实施加噪保护
        return self.he(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Server 持有后 2 层 + 分类头
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 辅助功能 (数据加载 & 反向防御 & 联邦聚合)
# ==========================================
class NEUDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try: img = Image.open(self.file_paths[idx]).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def load_non_iid_data(root):
    # 模拟 3 个工厂的数据孤岛
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    def get_loader(target_cls, batch_size=64):
        idx = [i for i, l in enumerate(labels) if l in target_cls]
        ds = NEUDataset([files[i] for i in idx], [labels[i] for i in idx], transform)
        return DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    test_idx = np.random.choice(len(files), min(300, len(files)), replace=False)
    test_loader = DataLoader(NEUDataset([files[i] for i in test_idx], [labels[i] for i in test_idx], transform), batch_size=64)
    
    return get_loader([0,1]), get_loader([2,3]), get_loader([4,5]), test_loader

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    """【保留】反向传播梯度防御"""
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def fed_avg(global_model, client_models, weights):
    """【升级】双向加权联邦聚合 (对齐论文 Bi-FedAvg)"""
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [
                c.state_dict()[k].float() * w 
                for c, w in zip(client_models, weights)
            ]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

# ==========================================
# 3. 终极流程：预训练 -> 联邦微调
# ==========================================
def run_perfect_pipeline():
    # --- A. 全量预训练 (CIFAR-10) ---
    print("\n☁️ [Stage 1] Pre-training Split-ViT on CIFAR-10 (Plaintext/GELU)...")
    
    g_client = ClientModel().to(DEVICE)
    g_server = ServerModel(num_classes=NUM_CLASSES_PRETRAIN).to(DEVICE)
    
    # 强制明文模式 (GELU, 无加噪)
    for m in g_client.modules():
        if isinstance(m, DynamicAct): m.mode = 'gelu'
        if isinstance(m, MockHE): m.noise_std = 0.0
    for m in g_server.modules():
        if isinstance(m, DynamicAct): m.mode = 'gelu'

    opt_c = optim.AdamW(g_client.parameters(), lr=1e-3)
    opt_s = optim.AdamW(g_server.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    tf_cifar = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    cifar_dl = DataLoader(datasets.CIFAR10('./data', train=True, download=True, transform=tf_cifar), batch_size=128, shuffle=True, num_workers=2)
    
    for ep in range(3):
        g_client.train(); g_server.train()
        for imgs, lbls in cifar_dl:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt_c.zero_grad(); opt_s.zero_grad()
            z = g_client(imgs)
            logits = g_server(z)
            loss = crit(logits, lbls)
            loss.backward()
            opt_c.step(); opt_s.step()
        print(f"   Pre-train Epoch {ep+1} Done.")
    
    torch.save(g_client.state_dict(), f'{SAVE_DIR}/pretrain_cifar_client.pth')
    torch.save(g_server.state_dict(), f'{SAVE_DIR}/pretrain_cifar_server.pth')
    print("✅ Stage 1 Complete. Models saved.")
    
    # --- B. 联邦学习准备 ---
    print("\n🏭 [Stage 2] Starting Federated Learning (Non-IID & Dual-Defense)...")
    
    g_server.head = nn.Linear(EMBED_DIM, NUM_CLASSES_FINETUNE).to(DEVICE)
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)] 
    
    # 请填入你真实的数据路径
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    loader_A, loader_B, loader_C, loader_test = load_non_iid_data(DATA_ROOT)
    loaders = [loader_A, loader_B, loader_C]
    
    # 计算各节点的联邦聚合权重
    sample_counts = [len(loader.dataset) for loader in loaders]
    total_samples = sum(sample_counts)
    client_weights = [count / total_samples for count in sample_counts]
    print(f"📊 Client Sample Counts: {sample_counts}")
    print(f"⚖️ Client Weights: {[f'{w:.4f}' for w in client_weights]}")

    # 开启联邦双向防御机制 (Poly激活, 引入噪声)
    for i in range(3):
        for m in clients[i].modules():
            if isinstance(m, DynamicAct): m.mode = 'poly'
            if isinstance(m, MockHE): m.noise_std = 1e-3 # 前向特征加噪
        for m in servers[i].modules():
            if isinstance(m, DynamicAct): m.mode = 'poly'
            
    # 全局模型也切换状态 (方便测试)
    for m in g_client.modules():
        if isinstance(m, DynamicAct): m.mode = 'poly'
        if isinstance(m, MockHE): m.noise_std = 1e-3
    for m in g_server.modules():
        if isinstance(m, DynamicAct): m.mode = 'poly'

    opts_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opts_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    
    # --- C. 联邦训练循环 ---
    best_acc = 0.0
    FED_ROUNDS = 50 
    
    for round_idx in range(1, FED_ROUNDS + 1):
        for i in range(3):
            clients[i].train(); servers[i].train()
            for imgs, labels in loaders[i]:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opts_c[i].zero_grad(); opts_s[i].zero_grad()
                
                # 1. 前向传播 (此时 Client 内部的 he 层会自动对特征 z 进行加噪)
                z = clients[i](imgs)
                z_payload = z.detach().clone().requires_grad_(True)
                logits = servers[i](z_payload)
                
                loss = crit(logits, labels)
                loss.backward()
                
                # 2. 反向传播防御 (梯度剪枝 + DP加噪)
                safe_grad = apply_gradient_defense(z_payload.grad, prune_ratio=0.5, noise_std=1e-3)
                z.backward(safe_grad)
                
                opts_c[i].step(); opts_s[i].step()
        
        # 3. 双向加权联邦聚合 (Bi-FedAvg)
        fed_avg(g_client, clients, client_weights)
        fed_avg(g_server, servers, client_weights)
        
        for i in range(3):
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())
            
        # 全局评估
        g_client.eval(); g_server.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in loader_test:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                pred = g_server(g_client(imgs)).argmax(1)
                correct += (pred == labels).sum().item()
                total += labels.size(0)
        acc = 100 * correct / total
        
        print(f"   Round {round_idx:02d} | 🏆 Global Acc: {acc:.2f}%")
        
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/fed_vit_client_best_v50.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/fed_vit_server_best_v50.pth')
            print(f"      🔥 New Best! Model Saved. ({best_acc:.2f}%)")

    print(f"\n✅ Experiment Finished. Final Best Accuracy: {best_acc:.2f}%")
    print(f"📂 Models are ready in: {SAVE_DIR}")

if __name__ == '__main__':
    run_perfect_pipeline()

🚀 Running on: cuda
📂 Models will be saved to: ./Fed_ViT_Final/

☁️ [Stage 1] Pre-training Split-ViT on CIFAR-10 (Plaintext/GELU)...
Files already downloaded and verified
   Pre-train Epoch 1 Done.
   Pre-train Epoch 2 Done.
   Pre-train Epoch 3 Done.
✅ Stage 1 Complete. Models saved.

🏭 [Stage 2] Starting Federated Learning (Non-IID & Dual-Defense)...
📊 Client Sample Counts: [600, 600, 600]
⚖️ Client Weights: ['0.3333', '0.3333', '0.3333']
   Round 01 | 🏆 Global Acc: 18.00%
      🔥 New Best! Model Saved. (18.00%)
   Round 02 | 🏆 Global Acc: 31.00%
      🔥 New Best! Model Saved. (31.00%)
   Round 03 | 🏆 Global Acc: 39.00%
      🔥 New Best! Model Saved. (39.00%)
   Round 04 | 🏆 Global Acc: 46.33%
      🔥 New Best! Model Saved. (46.33%)
   Round 05 | 🏆 Global Acc: 55.33%
      🔥 New Best! Model Saved. (55.33%)
   Round 06 | 🏆 Global Acc: 60.00%
      🔥 New Best! Model Saved. (60.00%)
   Round 07 | 🏆 Global Acc: 63.33%
      🔥 New Best! Model Saved. (63.33%)
   Round 08 | 🏆 Global Acc: 62.

In [6]:
# ==========================================
# 🛑 最终大结局：验证 88.33% 模型的密态推理能力
# ==========================================
import tenseal as ts
import numpy as np
import torch # 确保导入
import os # 确保导入

# 确保 ClientModel, ServerModel, PolyBlock, DynamicAct 类定义在内存中
# (如果你没重启内核，它们应该还在；如果重启了，需要重新运行 Cell 1)

def run_victory_lap_he_test():
    print("==================================================")
    print("🏆 最终大结局：验证的密态推理")
    print("==================================================")
    
    SAVE_DIR = 'Fed_ViT_Final' # 对应你刚才的保存路径
    C_PATH = f'{SAVE_DIR}/fed_vit_client_best_v50.pth'
    S_PATH = f'{SAVE_DIR}/fed_vit_server_best_v50.pth'
    
    if not os.path.exists(C_PATH):
        print("❌ 找不到模型文件！请检查目录。")
        return

    # 1. 加载模型
    client = ClientModel().to('cpu')
    server = ServerModel(num_classes=6).to('cpu')
    
    # 这一步最关键：看看加载的是不是那个高分模型
    client.load_state_dict(torch.load(C_PATH, map_location='cpu'))
    server.load_state_dict(torch.load(S_PATH, map_location='cpu'))
    client.eval(); server.eval()
    print("✅ 成功加载历史最佳模型 (Expected Acc: ~90.67%)")

    # 2. 准备数据 (随机抽一张)
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES'
    # 复用之前的 load 函数，只取 test_loader
    # 如果报错找不到函数，请重新运行定义 load_non_iid_data 的那个 cell
    _, _, _, test_loader = load_non_iid_data(DATA_ROOT)
    
    idx = np.random.randint(len(test_loader.dataset))
    img, label = test_loader.dataset[idx]
    img_tensor = img.unsqueeze(0)
    
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    true_label = CLASSES[label]
    print(f"📸 抽取测试样本 ID {idx}: 真实标签 = {true_label}")

    # 3. CKKS 密态推理
    print("\n🔐 启动 TenSEAL CKKS (PolyDegree=8192)...")
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60, 40, 40, 60])
    ctx.global_scale = 2**40
    ctx.generate_galois_keys()

    print("🚀 执行全流程密态推理...")
    with torch.no_grad():
        # Client 明文提取
        z_vec = client(img_tensor).mean(dim=1).flatten().numpy()
        
        # 加密
        enc_z = ts.ckks_vector(ctx, z_vec)
        
        # Cloud 密态计算 Head
        W = server.head.weight.data.numpy()
        b = server.head.bias.data.numpy()
        enc_logits = [enc_z.dot(W[i]) + float(b[i]) for i in range(6)]
        
        # 解密
        dec_logits = [e.decrypt()[0] for e in enc_logits]
        he_pred = CLASSES[np.argmax(dec_logits)]
        
        # 明文对照
        plain_logits = server.head(torch.tensor(z_vec))
        plain_pred = CLASSES[plain_logits.argmax().item()]

    # 4. 报告
    print("\n📊 [最终报告]")
    print(f"   真实标签: {true_label}")
    print(f"   明文预测: {plain_pred}")
    print(f"   密态预测: {he_pred}")
    
    mae = np.mean(np.abs(np.array(dec_logits) - plain_logits.numpy()))
    print(f"   🔒 误差(MAE): {mae:.8f}")

    if he_pred == plain_pred:
        print("\n✅ 验证通过！系统一致性完美。")
    else:
        print("\n⚠️ 存在偏差。")
        
    # 顺便画出最终的混淆矩阵图，作为论文配图
    # (此处省略绘图代码，如果你需要，可以直接运行之前给你的 plot_final_confusion_matrix)

run_victory_lap_he_test()

🏆 最终大结局：验证的密态推理
✅ 成功加载历史最佳模型 (Expected Acc: ~90.67%)
📸 抽取测试样本 ID 76: 真实标签 = patches

🔐 启动 TenSEAL CKKS (PolyDegree=8192)...
🚀 执行全流程密态推理...

📊 [最终报告]
   真实标签: patches
   明文预测: patches
   密态预测: patches
   🔒 误差(MAE): 0.00000102

✅ 验证通过！系统一致性完美。


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
from PIL import Image

# ==========================================
# 0. 全局配置与保存路径
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Fed_ViT_Final' # 📁 模型专门存放在这里
os.makedirs(SAVE_DIR, exist_ok=True)

# 确保实验可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
set_seed(42)

print(f"🚀 Running on: {DEVICE}")
print(f"📂 Models will be saved to: ./{SAVE_DIR}/")

# ViT 超参数
IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 # CIFAR-10
NUM_CLASSES_FINETUNE = 6  # NEU-DET

# ==========================================
# 1. 模型定义 (Split-ViT + HE适配双向防御)
# ==========================================
class DynamicAct(nn.Module):
    """
    动态激活函数 (融合第四章逻辑)：
    - mode='gelu': 正常训练，保证收敛
    - mode='poly': 模拟密文计算 (f(x) = ax^2 + bx + c)
    """
    def __init__(self):
        super().__init__()
        self.mode = 'gelu' # 初始默认为明文模式
        self.register_buffer('a', torch.tensor(0.17))
        self.register_buffer('b', torch.tensor(0.5))
        self.register_buffer('c', torch.tensor(0.12))

    def forward(self, x): 
        if self.mode == 'gelu':
            return nn.functional.gelu(x)
        return self.a * (x**2) + self.b * x + self.c

class MockHE(nn.Module):
    """
    【新增】前向特征防御：模拟同态加密层 (来自第四章)
    - 在特征 Z 上添加高斯噪声，模拟 CKKS 加密误差
    """
    def __init__(self):
        super().__init__()
        self.noise_std = 0.0 # 初始无噪声

    def forward(self, x):
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            return x + noise
        return x

class PolyBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), DynamicAct(), nn.Linear(dim * 2, dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        # Client 持有前 2 层
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        # 【新增】前向加密接口
        self.he = MockHE()

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        # 输出前经过 HE 层，开启后实施加噪保护
        return self.he(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Server 持有后 2 层 + 分类头
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 辅助功能 (数据加载 & 反向防御 & 联邦聚合)
# ==========================================
class NEUDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try: img = Image.open(self.file_paths[idx]).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def load_non_iid_data(root):
    # 模拟 3 个工厂的数据孤岛
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    def get_loader(target_cls, batch_size=64):
        idx = [i for i, l in enumerate(labels) if l in target_cls]
        ds = NEUDataset([files[i] for i in idx], [labels[i] for i in idx], transform)
        return DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    test_idx = np.random.choice(len(files), min(300, len(files)), replace=False)
    test_loader = DataLoader(NEUDataset([files[i] for i in test_idx], [labels[i] for i in test_idx], transform), batch_size=64)
    
    return get_loader([0,1]), get_loader([2,3]), get_loader([4,5]), test_loader

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    """【保留】反向传播梯度防御"""
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def fed_avg(global_model, client_models, weights):
    """【升级】双向加权联邦聚合 (对齐论文 Bi-FedAvg)"""
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [
                c.state_dict()[k].float() * w 
                for c, w in zip(client_models, weights)
            ]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)


def run_vanilla_fl_baseline():
    print("\n==================================================")
    print("🚀 启动消融实验：Vanilla 联邦明文训练 (无隐私保护)")
    print("   目的：剥离隐私机制，单纯评估 Non-IID 数据孤岛对精度的影响")
    print("==================================================")
    
    # --- A. 快速准备 (复用你已有的预训练权重以节省时间) ---
    print("\n☁️ [Stage 1] Loading Pre-trained Weights (CIFAR-10)...")
    g_client = ClientModel().to(DEVICE)
    g_server = ServerModel(num_classes=NUM_CLASSES_PRETRAIN).to(DEVICE)
    
    # 强制明文模式 (GELU, 无加噪)
    for m in g_client.modules():
        if isinstance(m, DynamicAct): m.mode = 'gelu'
        if isinstance(m, MockHE): m.noise_std = 0.0
    for m in g_server.modules():
        if isinstance(m, DynamicAct): m.mode = 'gelu'

    # 如果你本地已经有了刚才跑出来的预训练权重，直接加载，省去几分钟时间
    try:
        g_client.load_state_dict(torch.load(f'{SAVE_DIR}/pretrain_cifar_client.pth'))
        g_server.load_state_dict(torch.load(f'{SAVE_DIR}/pretrain_cifar_server.pth'))
        print("   ✅ Loaded existing pre-trained weights successfully.")
    except:
        print("   ⚠️ 未找到预训练权重，建议先运行 run_perfect_pipeline() 生成。")
        return

    # --- B. 联邦学习准备 ---
    print("\n🏭 [Stage 2] Starting Vanilla Federated Learning (Non-IID, Plaintext)...")
    
    # 换头
    g_server.head = nn.Linear(EMBED_DIM, NUM_CLASSES_FINETUNE).to(DEVICE)
    
    # 初始化三个孤岛节点
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)] 
    
    # 加载数据 (使用真实的路径)
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    loader_A, loader_B, loader_C, loader_test = load_non_iid_data(DATA_ROOT)
    loaders = [loader_A, loader_B, loader_C]
    
    # 计算 Bi-FedAvg 的聚合权重
    sample_counts = [len(loader.dataset) for loader in loaders]
    total_samples = sum(sample_counts)
    client_weights = [count / total_samples for count in sample_counts]
    print(f"📊 Client Sample Counts: {sample_counts}")
    
    # ！！！关键点：确保所有客户端和服务端都保持明文模式 ！！！
    for i in range(3):
        for m in clients[i].modules():
            if isinstance(m, DynamicAct): m.mode = 'gelu'
            if isinstance(m, MockHE): m.noise_std = 0.0 # 关闭前向特征加噪
        for m in servers[i].modules():
            if isinstance(m, DynamicAct): m.mode = 'gelu'

    opts_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opts_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    
    # --- C. 无隐私保护联邦训练循环 ---
    best_acc = 0.0
    FED_ROUNDS = 50 
    
    for round_idx in range(1, FED_ROUNDS + 1):
        for i in range(3):
            clients[i].train(); servers[i].train()
            for imgs, labels in loaders[i]:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opts_c[i].zero_grad(); opts_s[i].zero_grad()
                
                # 1. 前向传播 (此时输出的 z 没有任何扰动，原汁原味)
                z = clients[i](imgs)
                z_payload = z.detach().clone().requires_grad_(True)
                logits = servers[i](z_payload)
                
                loss = nn.CrossEntropyLoss()(logits, labels)
                loss.backward()
                
                # 2. 反向传播 (！！！关键区别：直接使用明文梯度，不经过防御函数！！！)
                z.backward(z_payload.grad) 
                
                opts_c[i].step(); opts_s[i].step()
        
        # 3. 双向加权联邦聚合 (依然保留聚合机制以对抗 Non-IID)
        fed_avg(g_client, clients, client_weights)
        fed_avg(g_server, servers, client_weights)
        
        for i in range(3):
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())
            
        # 全局评估
        g_client.eval(); g_server.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in loader_test:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                pred = g_server(g_client(imgs)).argmax(1)
                correct += (pred == labels).sum().item()
                total += labels.size(0)
        acc = 100 * correct / total
        
        print(f"   Round {round_idx:02d} | 🏆 Vanilla Global Acc: {acc:.2f}%")
        
        if acc > best_acc:
            best_acc = acc
            # 单独保存这个基准模型
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/vanilla_fed_client_best.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/vanilla_fed_server_best.pth')

    print(f"\n✅ Vanilla Baseline Finished. Final Best Accuracy: {best_acc:.2f}%")

if __name__ == '__main__':
    # 运行无隐私保护的消融实验基准
    run_vanilla_fl_baseline()

🚀 Running on: cuda
📂 Models will be saved to: ./Fed_ViT_Final/

🚀 启动消融实验：Vanilla 联邦明文训练 (无隐私保护)
   目的：剥离隐私机制，单纯评估 Non-IID 数据孤岛对精度的影响

☁️ [Stage 1] Loading Pre-trained Weights (CIFAR-10)...
   ✅ Loaded existing pre-trained weights successfully.

🏭 [Stage 2] Starting Vanilla Federated Learning (Non-IID, Plaintext)...
📊 Client Sample Counts: [600, 600, 600]
   Round 01 | 🏆 Vanilla Global Acc: 24.00%
   Round 02 | 🏆 Vanilla Global Acc: 37.00%
   Round 03 | 🏆 Vanilla Global Acc: 49.00%
   Round 04 | 🏆 Vanilla Global Acc: 59.67%
   Round 05 | 🏆 Vanilla Global Acc: 60.33%
   Round 06 | 🏆 Vanilla Global Acc: 64.33%
   Round 07 | 🏆 Vanilla Global Acc: 69.33%
   Round 08 | 🏆 Vanilla Global Acc: 71.67%
   Round 09 | 🏆 Vanilla Global Acc: 73.33%
   Round 10 | 🏆 Vanilla Global Acc: 74.67%
   Round 11 | 🏆 Vanilla Global Acc: 76.33%
   Round 12 | 🏆 Vanilla Global Acc: 77.67%
   Round 13 | 🏆 Vanilla Global Acc: 77.00%
   Round 14 | 🏆 Vanilla Global Acc: 82.00%
   Round 15 | 🏆 Vanilla Global Acc: 80.67

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
from PIL import Image

# ==========================================
# 0. 全局配置与保存路径
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Fed_ViT_Clean_Defense' # 📁 存储路径
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
set_seed(42)

print(f"🚀 Running on: {DEVICE}")

# ViT 超参数
IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  

# ==========================================
# 1. 模型组件 (Standard GELU + Forward Defense)
# ==========================================

class ForwardDefense(nn.Module):
    """
    【前向防御】中间特征加噪
    在客户端输出特征到服务端之前注入高斯噪声，防止服务端还原原始图像
    """
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std

    def forward(self, x):
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            return x + noise
        return x

class ViTBlock(nn.Module):
    """标准 Transformer Block (使用原始 GELU)"""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4), 
            nn.GELU(), 
            nn.Linear(dim * 4, dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        # Client 持有前 2 层
        self.blocks = nn.ModuleList([ViTBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0) # 默认为0，后续动态开启

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Server 持有后 2 层 + 分类头
        self.blocks = nn.ModuleList([ViTBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
        
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心算法 (后向防御 & 联邦聚合)
# ==========================================

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    """【后向防御】梯度剪枝 + DP 加噪"""
    if grad is None: return None
    g = grad.detach().clone()
    # 梯度剪枝
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    # 注入噪声
    return g + torch.randn_like(g) * noise_std

def fed_avg(global_model, client_models, weights):
    """双向加权联邦聚合 (Bi-FedAvg)"""
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [
                c.state_dict()[k].float() * w 
                for c, w in zip(client_models, weights)
            ]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

# ==========================================
# 3. 数据加载 (NEU-DET 模拟)
# ==========================================
class NEUDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        try: img = Image.open(self.file_paths[idx]).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def load_non_iid_data(root):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name:
                    files.append(path); labels.append(cls_map[c]); break
    
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    def get_loader(target_cls, batch_size=32):
        idx = [i for i, l in enumerate(labels) if l in target_cls]
        ds = NEUDataset([files[i] for i in idx], [labels[i] for i in idx], transform)
        return DataLoader(ds, batch_size=batch_size, shuffle=True)
    
    # 模拟三个厂区的非独立同分布数据
    return get_loader([0,1]), get_loader([2,3]), get_loader([4,5]), get_loader([0,1,2,3,4,5])

# ==========================================
# 4. 训练全流程
# ==========================================
def run_experiment():
    # --- A. 预训练 (CIFAR-10) ---
    print("\n☁️ [Stage 1] Pre-training on CIFAR-10 (Plaintext)...")
    g_client = ClientModel().to(DEVICE)
    g_server = ServerModel(num_classes=NUM_CLASSES_PRETRAIN).to(DEVICE)
    
    # 预训练不加噪以保证收敛
    g_client.defense.noise_std = 0.0
    
    # (假设此处已完成 CIFAR 训练并加载权重)
    # torch.save(g_client.state_dict(), 'pretrain_client.pth')
    
    # --- B. 联邦微调 (NEU-DET) ---
    print("\n🏭 [Stage 2] Starting Federated Fine-tuning with Dual-Defense...")
    g_server.head = nn.Linear(EMBED_DIM, NUM_CLASSES_FINETUNE).to(DEVICE)
    
    # 实例化 3 个客户端节点
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    
    # 开启前向防御：注入噪声
    for c in clients: c.defense.noise_std = 0.001 
    
    # 数据准备 (请确保 DATA_ROOT 路径正确)
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    loader_A, loader_B, loader_C, loader_test = load_non_iid_data(DATA_ROOT)
    loaders = [loader_A, loader_B, loader_C]
    
    client_weights = [len(l.dataset) for l in loaders]
    total = sum(client_weights)
    client_weights = [w/total for w in client_weights]

    opts_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opts_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()

    # --- C. 联邦训练循环 ---
    for round_idx in range(1, 51):
        for i in range(3):
            clients[i].train(); servers[i].train()
            for imgs, labels in loaders[i]:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opts_c[i].zero_grad(); opts_s[i].zero_grad()
                
                # 1. 前向传播 (含 Client 侧特征加噪)
                z = clients[i](imgs)
                z_payload = z.detach().clone().requires_grad_(True)
                logits = servers[i](z_payload)
                
                loss = crit(logits, labels)
                loss.backward()
                
                # 2. 反向传播防御 (Server 侧对回传梯度加噪剪枝)
                safe_grad = apply_gradient_defense(z_payload.grad, prune_ratio=0.5, noise_std=1e-3)
                z.backward(safe_grad)
                
                opts_c[i].step(); opts_s[i].step()

        # 3. 聚合
        fed_avg(g_client, clients, client_weights)
        fed_avg(g_server, servers, client_weights)
        
        # 同步
        for i in range(3):
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())

        # 4. 评估
        g_client.eval(); g_server.eval()
        correct, total_test = 0, 0
        with torch.no_grad():
            for imgs, labels in loader_test:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = g_server(g_client(imgs))
                correct += (out.argmax(1) == labels).sum().item()
                total_test += labels.size(0)
        
        print(f"Round {round_idx:02d} | 🏆 Acc: {100*correct/total_test:.2f}%")

if __name__ == '__main__':
    run_experiment()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Fed_Final_Results' # 📁 存放所有 CSV 表格和 PTH 模型的文件夹
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

# 超参数设置
IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES = 6 
FED_ROUNDS = 50  # ⬅️ 正式设为 50 轮

# ==========================================
# 1. 核心模型与双向防御定义
# ==========================================
class ForwardDefense(nn.Module):
    """【前向防御】在边缘端特征流出前加噪"""
    def __init__(self, std=0.001):
        super().__init__()
        self.noise_std = std

    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    """标准 ViT Block (原生 GELU, 4倍MLP)"""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.001)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    """【后向防御】云端下发梯度前：剪枝 + DP 加噪"""
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def fed_avg(global_model, client_models, weights):
    """加权联邦聚合"""
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

# ==========================================
# 2. 三合一数据分布生成器
# ==========================================
class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', alpha=1.0, num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
                    
    files, labels = np.array(files), np.array(labels)
    
    # 划分全局测试集 (20%)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8)
    train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    
    train_files, train_labels = files[train_idx], labels[train_idx]
    test_files, test_labels = files[test_idx], labels[test_idx]
    
    client_files = [[] for _ in range(num_clients)]
    client_labels = [[] for _ in range(num_clients)]
    
    if mode == 'iid':
        splits = np.array_split(np.random.permutation(len(train_files)), num_clients)
        for i in range(num_clients):
            client_files[i] = train_files[splits[i]].tolist()
            client_labels[i] = train_labels[splits[i]].tolist()
            
    elif mode == 'weak_non_iid':
        for c in range(NUM_CLASSES):
            idx_c = np.where(train_labels == c)[0]
            np.random.shuffle(idx_c)
            proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
            proportions = np.cumsum(proportions) * len(idx_c)
            splits = np.split(idx_c, proportions.astype(int)[:-1])
            for i in range(num_clients):
                client_files[i].extend(train_files[splits[i]].tolist())
                client_labels[i].extend(train_labels[splits[i]].tolist())

    elif mode == 'pathological':
        target_classes = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx_c = [idx for idx, label in enumerate(train_labels) if label in target_classes[i]]
            client_files[i] = train_files[idx_c].tolist()
            client_labels[i] = train_labels[idx_c].tolist()

    print(f"\n📊 Data Distribution Mode: {mode.upper()}")
    for i in range(num_clients):
        cls_counts = [client_labels[i].count(c) for c in range(NUM_CLASSES)]
        print(f"  Client {i+1} class counts: {cls_counts}")

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), 
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), 
        transforms.Normalize((0.5,), (0.5,))
    ])
    test_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), 
        transforms.ToTensor(), 
        transforms.Normalize((0.5,), (0.5,))
    ])
    
    loaders = [DataLoader(NEUDataset(client_files[i], client_labels[i], transform), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_files.tolist(), test_labels.tolist(), test_transform), batch_size=64, shuffle=False)
    
    return loaders, test_loader

# ==========================================
# 3. 实验流程 (带 CSV 记录与最佳模型保存)
# ==========================================
def run_fl_experiment(mode, data_root, rounds=50):
    loaders, loader_test = generate_fl_data(data_root, mode=mode)
    
    # 动态计算各节点聚合权重
    client_weights = [len(l.dataset) for l in loaders]
    total_samples = sum(client_weights)
    client_weights = [w / total_samples for w in client_weights]

    g_client = ClientModel().to(DEVICE)
    g_server = ServerModel().to(DEVICE)
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]

    opts_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opts_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()

    history = {'loss': [], 'acc': []}
    best_acc = 0.0  # 用于记录全局最佳精度
    
    print(f"\n🚀 Starting {mode.upper()} Federated Training for {rounds} Rounds...")
    for round_idx in range(1, rounds + 1):
        round_losses = []
        for i in range(3):
            clients[i].train(); servers[i].train()
            epoch_loss = 0.0
            
            if len(loaders[i]) == 0: continue 
            
            for imgs, labels in loaders[i]:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                opts_c[i].zero_grad(); opts_s[i].zero_grad()
                
                # SFL 训练流
                z = clients[i](imgs)
                z_payload = z.detach().clone().requires_grad_(True)
                logits = servers[i](z_payload)
                loss = crit(logits, labels)
                
                loss.backward()
                safe_grad = apply_gradient_defense(z_payload.grad, prune_ratio=0.5, noise_std=1e-3)
                z.backward(safe_grad)
                
                opts_c[i].step(); opts_s[i].step()
                epoch_loss += loss.item()
            round_losses.append(epoch_loss / len(loaders[i]))
            
        avg_loss = sum(round_losses) / len(round_losses)
        history['loss'].append(avg_loss)

        # 全局聚合与同步
        fed_avg(g_client, clients, client_weights)
        fed_avg(g_server, servers, client_weights)
        for i in range(3):
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())

        # 全局评估
        g_client.eval(); g_server.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in loader_test:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                pred = g_server(g_client(imgs)).argmax(1)
                correct += (pred == labels).sum().item()
                total += labels.size(0)
        
        acc = 100 * correct / total
        history['acc'].append(acc)
        print(f"  Round {round_idx:02d}/{rounds} | Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

        # 保存最佳模型权重
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}.pth')
            print(f"      🔥 New Best Saved! ({best_acc:.2f}%)")

    # 训练结束，将 Loss 和 Acc 写入 CSV 表格
    csv_path = f'{SAVE_DIR}/history_{mode}.csv'
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Round', 'Loss', 'Accuracy'])
        for r, (l, a) in enumerate(zip(history['loss'], history['acc']), 1):
            writer.writerow([r, round(l, 4), round(a, 2)])
            
    print(f"✅ {mode.upper()} experiment finished. Data saved to {csv_path}")

if __name__ == '__main__':
    # ⚠️ 请确保替换为你服务器上的真实数据路径
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    
    # 依次运行三种不同数据分布的联邦实验
    run_fl_experiment(mode='iid', data_root=DATA_ROOT, rounds=FED_ROUNDS)
    run_fl_experiment(mode='weak_non_iid', data_root=DATA_ROOT, rounds=FED_ROUNDS)
    run_fl_experiment(mode='pathological', data_root=DATA_ROOT, rounds=FED_ROUNDS)
    
    print(f"\n🎉 All {FED_ROUNDS}-round experiments are fully completed!")
    print(f"📂 Please check the './{SAVE_DIR}/' folder for CSV charts and best Model Weights.")

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Final_FL_Experiments_Results' 
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

# ==========================================
# 1. 模型定义 (Standard ViT)
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心逻辑 (聚合、数据、防御)
# ==========================================
def fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', alpha=1.0, num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8); train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    train_f, train_l = files[train_idx], labels[train_idx]
    test_f, test_l = files[test_idx], labels[test_idx]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    if mode == 'iid':
        s = np.array_split(np.random.permutation(len(train_f)), num_clients)
        for i in range(num_clients): cf[i], cl[i] = train_f[s[i]].tolist(), train_l[s[i]].tolist()
    elif mode == 'weak_non_iid':
        for c in range(NUM_CLASSES_FINETUNE):
            idx_c = np.where(train_l == c)[0]; np.random.shuffle(idx_c)
            prop = np.cumsum(np.random.dirichlet(np.repeat(alpha, num_clients))) * len(idx_c)
            splits = np.split(idx_c, prop.astype(int)[:-1])
            for i in range(num_clients): cf[i].extend(train_f[splits[i]].tolist()); cl[i].extend(train_l[splits[i]].tolist())
    elif mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 3. 实验运行核心
# ==========================================

def train_pretrain():
    print("\n" + "★"*40 + "\n[阶段 1] CIFAR-10 集中式预训练开启...\n" + "★"*40)
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(10):
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls); loss.backward(); opt.step(); loss_sum += loss.item()
        print(f"  > Pretrain Epoch {ep+1:02d} | Avg Loss: {loss_sum/len(loader):.4f}")
    return c_m, s_m

def run_fl_experiment(mode, data_root, use_defense, pretrain_weights, exp_idx):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*60)
    print(f"💠 [实验 {exp_idx}/6] 分布: {mode.upper()} | 隐私防御: {def_str.upper()}")
    print(f"  >> 预载权重: 已加载 | 总轮次: {FED_ROUNDS}")
    print("="*60)
    
    loaders, test_loader = generate_fl_data(data_root, mode=mode)
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 打印每个客户端的数据分布
    for i in range(3):
        print(f"  - Client {i+1} 样本数: {len(loaders[i].dataset)}")

    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        r_losses = []
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        fed_avg(g_client, clients, c_weights); fed_avg(g_server, servers, c_weights)
        for i in range(3): clients[i].load_state_dict(g_client.state_dict()); servers[i].load_state_dict(g_server.state_dict())
        
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}%"
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}_{def_str}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}_{def_str}.pth')
            status += " ⭐ [Model Saved]"
        print(status)

    csv_fn = f'history_{mode}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验完成! 记录已存至: {csv_fn}")

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == '__main__':
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' # ⚠️请确保路径正确
    
    # 1. 预训练
    pre_c, pre_s = train_pretrain()
    
    # 2. 消融实验
    exp_count = 1
    for dist in ['iid', 'weak_non_iid', 'pathological']:
        for defense in [False, True]:
            run_fl_experiment(dist, DATA_ROOT, defense, (pre_c, pre_s), exp_count)
            exp_count += 1
            
    print("\n" + "✅"*20 + "\n所有实验已全部圆满结束！\n" + "✅"*20)
    print(f"数据与模型全部存放在文件夹: ./{SAVE_DIR}/")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练开启...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572
  > Pretrain Epoch 02 | Avg Loss: 1.6377
  > Pretrain Epoch 03 | Avg Loss: 1.5811
  > Pretrain Epoch 04 | Avg Loss: 1.5440
  > Pretrain Epoch 05 | Avg Loss: 1.5150
  > Pretrain Epoch 06 | Avg Loss: 1.4821
  > Pretrain Epoch 07 | Avg Loss: 1.4387
  > Pretrain Epoch 08 | Avg Loss: 1.3904
  > Pretrain Epoch 09 | Avg Loss: 1.3975
  > Pretrain Epoch 10 | Avg Loss: 1.3675

💠 [实验 1/6] 分布: IID | 隐私防御: NO_DEFENSE
  >> 预载权重: 已加载 | 总轮次: 50
  - Client 1 样本数: 480
  - Client 2 样本数: 480
  - Client 3 样本数: 480
  Round 01/50 | Loss: 1.7968 | Acc: 25.28% ⭐ [Model Saved]
  Round 02/50 | Loss: 1.5276 | Acc: 57.22% ⭐ [Model Saved]
  Round 03/50 | Loss: 1.3357 | Acc: 63.61% ⭐ [Model Saved]
  Round 04/50 | Loss: 1.1763 | Acc: 67.50% ⭐ [Model Saved]
  Round 05/50 | Loss: 1.0414 | Acc: 71.67% ⭐ [Model Saved]
  Round 06/50 | 

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Final_FL_Experiments_Results_pre3' 
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

# ==========================================
# 1. 模型定义 (Standard ViT)
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心逻辑 (聚合、数据、防御)
# ==========================================
def fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', alpha=1.0, num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8); train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    train_f, train_l = files[train_idx], labels[train_idx]
    test_f, test_l = files[test_idx], labels[test_idx]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    if mode == 'iid':
        s = np.array_split(np.random.permutation(len(train_f)), num_clients)
        for i in range(num_clients): cf[i], cl[i] = train_f[s[i]].tolist(), train_l[s[i]].tolist()
    elif mode == 'weak_non_iid':
        for c in range(NUM_CLASSES_FINETUNE):
            idx_c = np.where(train_l == c)[0]; np.random.shuffle(idx_c)
            prop = np.cumsum(np.random.dirichlet(np.repeat(alpha, num_clients))) * len(idx_c)
            splits = np.split(idx_c, prop.astype(int)[:-1])
            for i in range(num_clients): cf[i].extend(train_f[splits[i]].tolist()); cl[i].extend(train_l[splits[i]].tolist())
    elif mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 3. 实验运行核心
# ==========================================

def train_pretrain():
    print("\n" + "★"*40 + "\n[阶段 1] CIFAR-10 集中式预训练开启...\n" + "★"*40)
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(3):
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls); loss.backward(); opt.step(); loss_sum += loss.item()
        print(f"  > Pretrain Epoch {ep+1:02d} | Avg Loss: {loss_sum/len(loader):.4f}")
    return c_m, s_m

def run_fl_experiment(mode, data_root, use_defense, pretrain_weights, exp_idx):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*60)
    print(f"💠 [实验 {exp_idx}/6] 分布: {mode.upper()} | 隐私防御: {def_str.upper()}")
    print(f"  >> 预载权重: 已加载 | 总轮次: {FED_ROUNDS}")
    print("="*60)
    
    loaders, test_loader = generate_fl_data(data_root, mode=mode)
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 打印每个客户端的数据分布
    for i in range(3):
        print(f"  - Client {i+1} 样本数: {len(loaders[i].dataset)}")

    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        r_losses = []
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        fed_avg(g_client, clients, c_weights); fed_avg(g_server, servers, c_weights)
        for i in range(3): clients[i].load_state_dict(g_client.state_dict()); servers[i].load_state_dict(g_server.state_dict())
        
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}%"
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}_{def_str}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}_{def_str}.pth')
            status += " ⭐ [Model Saved]"
        print(status)

    csv_fn = f'history_{mode}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验完成! 记录已存至: {csv_fn}")

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == '__main__':
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' # ⚠️请确保路径正确
    
    # 1. 预训练
    pre_c, pre_s = train_pretrain()
    
    # 2. 消融实验
    exp_count = 1
    for dist in ['iid', 'weak_non_iid', 'pathological']:
        for defense in [False, True]:
            run_fl_experiment(dist, DATA_ROOT, defense, (pre_c, pre_s), exp_count)
            exp_count += 1
            
    print("\n" + "✅"*20 + "\n所有实验已全部圆满结束！\n" + "✅"*20)
    print(f"数据与模型全部存放在文件夹: ./{SAVE_DIR}/")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练开启...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572
  > Pretrain Epoch 02 | Avg Loss: 1.6421
  > Pretrain Epoch 03 | Avg Loss: 1.5805

💠 [实验 1/6] 分布: IID | 隐私防御: NO_DEFENSE
  >> 预载权重: 已加载 | 总轮次: 50
  - Client 1 样本数: 480
  - Client 2 样本数: 480
  - Client 3 样本数: 480
  Round 01/50 | Loss: 1.6752 | Acc: 37.50% ⭐ [Model Saved]
  Round 02/50 | Loss: 1.4234 | Acc: 54.72% ⭐ [Model Saved]
  Round 03/50 | Loss: 1.2535 | Acc: 62.22% ⭐ [Model Saved]
  Round 04/50 | Loss: 1.1129 | Acc: 64.17% ⭐ [Model Saved]
  Round 05/50 | Loss: 0.9884 | Acc: 67.50% ⭐ [Model Saved]
  Round 06/50 | Loss: 0.8855 | Acc: 71.11% ⭐ [Model Saved]
  Round 07/50 | Loss: 0.8051 | Acc: 73.61% ⭐ [Model Saved]
  Round 08/50 | Loss: 0.7366 | Acc: 78.06% ⭐ [Model Saved]
  Round 09/50 | Loss: 0.6675 | Acc: 80.56% ⭐ [Model Saved]
  Round 10/50 | Loss: 0.6283 | Acc: 81.67% ⭐ [Model Saved]
  Round 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Final_FL_Experiments_strict_pre10' 
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

# ==========================================
# 1. 模型定义 (Standard ViT)
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心逻辑 (聚合、数据、防御)
# ==========================================
def fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', alpha=1.0, num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8); train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    train_f, train_l = files[train_idx], labels[train_idx]
    test_f, test_l = files[test_idx], labels[test_idx]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    if mode == 'iid':
        s = np.array_split(np.random.permutation(len(train_f)), num_clients)
        for i in range(num_clients): cf[i], cl[i] = train_f[s[i]].tolist(), train_l[s[i]].tolist()
    elif mode == 'weak_non_iid':
        for c in range(NUM_CLASSES_FINETUNE):
            idx_c = np.where(train_l == c)[0]; np.random.shuffle(idx_c)
            prop = np.cumsum(np.random.dirichlet(np.repeat(alpha, num_clients))) * len(idx_c)
            splits = np.split(idx_c, prop.astype(int)[:-1])
            for i in range(num_clients): cf[i].extend(train_f[splits[i]].tolist()); cl[i].extend(train_l[splits[i]].tolist())
    elif mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 3. 实验运行核心
# ==========================================

def train_pretrain():
    print("\n" + "★"*40 + "\n[阶段 1] CIFAR-10 集中式预训练开启...\n" + "★"*40)
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(10):
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls); loss.backward(); opt.step(); loss_sum += loss.item()
        print(f"  > Pretrain Epoch {ep+1:02d} | Avg Loss: {loss_sum/len(loader):.4f}")
    return c_m, s_m

'''
def run_fl_experiment(mode, data_root, use_defense, pretrain_weights, exp_idx):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*60)
    print(f"💠 [实验 {exp_idx}/6] 分布: {mode.upper()} | 隐私防御: {def_str.upper()}")
    print(f"  >> 预载权重: 已加载 | 总轮次: {FED_ROUNDS}")
    print("="*60)
    
    loaders, test_loader = generate_fl_data(data_root, mode=mode)
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 打印每个客户端的数据分布
    for i in range(3):
        print(f"  - Client {i+1} 样本数: {len(loaders[i].dataset)}")
'''

# ⬇️ 修改参数：把 data_root 改为直接接收 loaders 和 test_loader
def run_fl_experiment(mode, loaders, test_loader, use_defense, pretrain_weights, exp_idx):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*60)
    print(f"💠 [实验 {exp_idx}/6] 分布: {mode.upper()} | 隐私防御: {def_str.upper()}")
    print(f"  >> 预载权重: 已加载 | 总轮次: {FED_ROUNDS}")
    print("="*60)
    
    # ⬇️ 这里不再重新生成数据了，直接使用外面传进来的
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 打印每个客户端的数据分布 (你会发现带防御和不带防御时，这里打印的数字将完全一样！)
    for i in range(3):
        print(f"  - Client {i+1} 样本数: {len(loaders[i].dataset)}")

    # ... [下方的所有模型训练、评估、保存代码完全保持不变] ...

    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        r_losses = []
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        fed_avg(g_client, clients, c_weights); fed_avg(g_server, servers, c_weights)
        for i in range(3): clients[i].load_state_dict(g_client.state_dict()); servers[i].load_state_dict(g_server.state_dict())
        
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}%"
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}_{def_str}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}_{def_str}.pth')
            status += " ⭐ [Model Saved]"
        print(status)

    csv_fn = f'history_{mode}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验完成! 记录已存至: {csv_fn}")

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == '__main__':
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' # ⚠️请确保路径正确
    
    # 1. 预训练
    pre_c, pre_s = train_pretrain()
    
    # 2. 消融实验
    exp_count = 1
    for dist in ['iid', 'weak_non_iid', 'pathological']:
        
        # 🌟 核心修复点：在这里生成数据！
        # 这样就能保证这个分布下的有/无防御组，吃的是完完全全同一盘菜。
        current_loaders, current_test_loader = generate_fl_data(DATA_ROOT, mode=dist)
        
        for defense in [False, True]:
            # 把生成好的 current_loaders 和 current_test_loader 传进去
            run_fl_experiment(dist, current_loaders, current_test_loader, defense, (pre_c, pre_s), exp_count)
            exp_count += 1
            
    print("\n" + "✅"*20 + "\n所有实验已全部圆满结束！\n" + "✅"*20)
    print(f"数据与模型全部存放在文件夹: ./{SAVE_DIR}/")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练开启...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572
  > Pretrain Epoch 02 | Avg Loss: 1.6320
  > Pretrain Epoch 03 | Avg Loss: 1.5642
  > Pretrain Epoch 04 | Avg Loss: 1.5194
  > Pretrain Epoch 05 | Avg Loss: 1.4818
  > Pretrain Epoch 06 | Avg Loss: 1.4457
  > Pretrain Epoch 07 | Avg Loss: 1.4156
  > Pretrain Epoch 08 | Avg Loss: 1.3999
  > Pretrain Epoch 09 | Avg Loss: 1.3671
  > Pretrain Epoch 10 | Avg Loss: 1.3593

💠 [实验 1/6] 分布: IID | 隐私防御: NO_DEFENSE
  >> 预载权重: 已加载 | 总轮次: 50
  - Client 1 样本数: 480
  - Client 2 样本数: 480
  - Client 3 样本数: 480
  Round 01/50 | Loss: 1.7949 | Acc: 31.11% ⭐ [Model Saved]
  Round 02/50 | Loss: 1.5294 | Acc: 53.06% ⭐ [Model Saved]
  Round 03/50 | Loss: 1.3399 | Acc: 63.33% ⭐ [Model Saved]
  Round 04/50 | Loss: 1.1817 | Acc: 69.44% ⭐ [Model Saved]
  Round 05/50 | Loss: 1.0417 | Acc: 75.56% ⭐ [Model Saved]
  Round 06/50 | 

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Pretrain_Ablation_Results' 
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

# ==========================================
# 1. 模型架构定义
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心算法 (防御、聚合、数据)
# ==========================================
def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
                    
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8)
    train_f, train_l = files[indices[:split_idx]], labels[indices[:split_idx]]
    test_f, test_l = files[indices[split_idx:]], labels[indices[split_idx:]]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    
    if mode == 'iid':
        splits = np.array_split(np.random.permutation(len(train_f)), num_clients)
        for i in range(num_clients): 
            cf[i], cl[i] = train_f[splits[i]].tolist(), train_l[splits[i]].tolist()
            
    elif mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    
    return loaders, test_loader

# ==========================================
# 3. 优化版预训练：沿途保存 Checkpoints
# ==========================================
def train_pretrain_checkpoints(target_epochs=[3, 5, 10, 20]):
    max_epochs = max(target_epochs)
    print("\n" + "★"*50)
    print(f"[阶段 1] CIFAR-10 集中式预训练 (共 {max_epochs} 轮)")
    print(f"目标检查点: {target_epochs}")
    print("★"*50)
    
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    checkpoints = {}
    
    for ep in range(1, max_epochs + 1):
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad()
            loss = crit(s_m(c_m(imgs)), lbls)
            loss.backward()
            opt.step()
            loss_sum += loss.item()
            
        print(f"  > Pretrain Epoch {ep:02d} | Avg Loss: {loss_sum/len(loader):.4f}")
        
        # 🌟 沿途下蛋：到达指定轮次，立即保存深拷贝
        if ep in target_epochs:
            checkpoints[ep] = (copy.deepcopy(c_m), copy.deepcopy(s_m))
            print(f"    📦 已保存预训练权重: {ep} 轮")
            
    return checkpoints

# ==========================================
# 4. 联邦消融实验核心
# ==========================================
def run_fl_experiment(mode, loaders, test_loader, use_defense, pretrain_weights, pre_ep, exp_idx, total_exp):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*65)
    print(f"💠 [实验 {exp_idx}/{total_exp}] 分布: {mode.upper()} | 预训练: {pre_ep} 轮 | 防御: {def_str.upper()}")
    print("="*65)
    
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 初始化客户端与服务端 (加载对应轮次的预训练权重)
    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        r_losses = []
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        fed_avg(g_client, clients, c_weights); fed_avg(g_server, servers, c_weights)
        for i in range(3): clients[i].load_state_dict(g_client.state_dict()); servers[i].load_state_dict(g_server.state_dict())
        
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}%"
        if acc > best_acc:
            best_acc = acc
            # 文件名加入预训练轮数标识，例如: best_client_iid_pre10_with_defense.pth
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}_pre{pre_ep}_{def_str}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}_pre{pre_ep}_{def_str}.pth')
            status += " ⭐ [Best Saved]"
        print(status)

    csv_fn = f'history_{mode}_pre{pre_ep}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验 {exp_idx} 完成! 最佳精度: {best_acc:.2f}% | 记录已存至: {csv_fn}")

# ==========================================
# 5. 主程序流控制
# ==========================================
if __name__ == '__main__':
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' # ⚠️替换为真实路径
    
    pre_epochs_list = [3, 5, 10, 20]
    distributions = ['iid', 'pathological']
    defenses = [False, True]
    
    total_experiments = len(distributions) * len(pre_epochs_list) * len(defenses) # 2 * 4 * 2 = 16
    
    # 1. 预训练并收集 4 个阶段的检查点权重
    pretrain_checkpoints = train_pretrain_checkpoints(target_epochs=pre_epochs_list)
    
    # 2. 正式开展 16 组消融实验
    exp_count = 1
    for dist in distributions:
        
        # 🌟 同一分布下，所有预训练轮数、所有防御模式，共用这一批绝对一致的数据！
        print(f"\n📦 正在切割并锁定 [{dist.upper()}] 分布的训练/测试集...")
        current_loaders, current_test_loader = generate_fl_data(DATA_ROOT, mode=dist)
        
        for pre_ep in pre_epochs_list:
            for defense in defenses:
                # 获取对应预训练轮次的权重
                weights_for_this_exp = pretrain_checkpoints[pre_ep]
                
                run_fl_experiment(
                    mode=dist, 
                    loaders=current_loaders, 
                    test_loader=current_test_loader, 
                    use_defense=defense, 
                    pretrain_weights=weights_for_this_exp, 
                    pre_ep=pre_ep, 
                    exp_idx=exp_count, 
                    total_exp=total_experiments
                )
                exp_count += 1
                
    print("\n" + "✅"*25 + "\n所有 16 组实验已全部完美通过！\n" + "✅"*25)
    print(f"请检查文件夹 ./{SAVE_DIR}/，共计 16个CSV表格 和 32个PTH模型。")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练 (共 20 轮)
目标检查点: [3, 5, 10, 20]
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572
  > Pretrain Epoch 02 | Avg Loss: 1.6324
  > Pretrain Epoch 03 | Avg Loss: 1.5682
    📦 已保存预训练权重: 3 轮
  > Pretrain Epoch 04 | Avg Loss: 1.5185
  > Pretrain Epoch 05 | Avg Loss: 1.4699
    📦 已保存预训练权重: 5 轮
  > Pretrain Epoch 06 | Avg Loss: 1.4352
  > Pretrain Epoch 07 | Avg Loss: 1.4172
  > Pretrain Epoch 08 | Avg Loss: 1.3917
  > Pretrain Epoch 09 | Avg Loss: 1.3501
  > Pretrain Epoch 10 | Avg Loss: 1.3413
    📦 已保存预训练权重: 10 轮
  > Pretrain Epoch 11 | Avg Loss: 1.3218
  > Pretrain Epoch 12 | Avg Loss: 1.2977
  > Pretrain Epoch 13 | Avg Loss: 1.2959
  > Pretrain Epoch 14 | Avg Loss: 1.2654
  > Pretrain Epoch 15 | Avg Loss: 1.2343
  > Pretrain Epoch 16 | Avg Loss: 1.2142
  > Pretrain Epoch 17 | Avg Loss: 1.1817
  > Pretrain Epoch 18 | Avg Loss: 1.1451
  > Pr

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Agg_Mode_Ablation_Results' 
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

# ==========================================
# 1. 模型定义 (Standard ViT)
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 2. 核心逻辑 (聚合、数据、防御)
# ==========================================
def fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='pathological', num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8)
    train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    train_f, train_l = files[train_idx], labels[train_idx]
    test_f, test_l = files[test_idx], labels[test_idx]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    
    if mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 3. 实验运行核心
# ==========================================
def train_pretrain():
    print("\n" + "★"*40 + "\n[阶段 1] CIFAR-10 集中式预训练 (10轮)...\n" + "★"*40)
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(10):
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls); loss.backward(); opt.step(); loss_sum += loss.item()
        print(f"  > Pretrain Epoch {ep+1:02d} | Avg Loss: {loss_sum/len(loader):.4f}")
    return c_m, s_m

def run_architecture_experiment(agg_mode, loaders, test_loader, use_defense, pretrain_weights, exp_idx):
    """
    agg_mode: 'bidirectional' (双向聚合) 或 'unidirectional' (单向共享 Server)
    """
    mode_str = "BIDIRECTIONAL (双向)" if agg_mode == 'bidirectional' else "UNIDIRECTIONAL (单向)"
    def_str = "WITH_DEFENSE" if use_defense else "NO_DEFENSE"
    
    print(f"\n" + "="*65)
    print(f"💠 [实验 {exp_idx}/4] 架构: {mode_str} | 防御: {def_str}")
    print("="*65)
    
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    # 初始化全局基座模型
    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    # 边缘端 (Client) 始终是独立的副本
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    
    # 🌟 核心差异：云端 (Server) 架构区分
    if agg_mode == 'bidirectional':
        servers = [copy.deepcopy(g_server) for _ in range(3)]
        opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    else:
        # 单向聚合：云端只有一个共享的 Server 模型
        shared_server = copy.deepcopy(g_server)
        opt_shared_s = optim.AdamW(shared_server.parameters(), lr=1e-4)
        
    crit = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        r_losses = []
        
        # --- 本地训练阶段 ---
        for i in range(3):
            clients[i].train()
            # 动态选择 Server 实例和优化器
            current_server = servers[i] if agg_mode == 'bidirectional' else shared_server
            current_opt_s = opt_s[i] if agg_mode == 'bidirectional' else opt_shared_s
            current_server.train()
            
            ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); current_opt_s.zero_grad()
                
                z = clients[i](imgs)
                zp = z.detach().clone().requires_grad_(True)
                out = current_server(zp)
                loss = crit(out, lbls)
                loss.backward()
                
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g)
                
                opt_c[i].step(); current_opt_s.step()
                ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        
        # --- 联邦聚合阶段 ---
        fed_avg(g_client, clients, c_weights)
        for i in range(3): clients[i].load_state_dict(g_client.state_dict())
        
        if agg_mode == 'bidirectional':
            # 双向：独立聚合 Server
            fed_avg(g_server, servers, c_weights)
            for i in range(3): servers[i].load_state_dict(g_server.state_dict())
        else:
            # 单向：Server 已经是共享状态，直接用于测试
            g_server.load_state_dict(shared_server.state_dict())
            
        # --- 测试阶段 ---
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}%"
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{agg_mode}_{def_str}.pth')
            status += " ⭐ [Model Saved]"
        print(status)

    csv_fn = f'history_{agg_mode}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验完成! 记录已存至: {csv_fn}")

# ==========================================
# 4. 主程序
# ==========================================
if __name__ == '__main__':
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' # ⚠️ 确保路径正确
    
    # 1. 执行 10 轮预训练
    pre_c, pre_s = train_pretrain()
    
    # 2. 生成锁定的一次性 Pathological 数据，供 4 组实验共用！
    print("\n📦 正在切割并锁定 [PATHOLOGICAL] 强非独立同分布数据...")
    current_loaders, current_test_loader = generate_fl_data(DATA_ROOT, mode='pathological')
    for i in range(3):
        print(f"  - Client {i+1} 样本数: {len(current_loaders[i].dataset)}")
    
    # 3. 开启 4 组对比消融
    configs = [
        ('bidirectional', False), # 实验 1: 双向明文
        ('unidirectional', False),# 实验 2: 单向明文
        ('bidirectional', True),  # 实验 3: 双向 + 隐私防护
        ('unidirectional', True)  # 实验 4: 单向 + 隐私防护
    ]
    
    for idx, (mode, defense) in enumerate(configs, 1):
        run_architecture_experiment(mode, current_loaders, current_test_loader, defense, (pre_c, pre_s), idx)
            
    print("\n" + "✅"*20 + "\n所有架构消融实验已全部圆满结束！\n" + "✅"*20)
    print(f"数据与模型全部存放在文件夹: ./{SAVE_DIR}/")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练 (10轮)...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572
  > Pretrain Epoch 02 | Avg Loss: 1.6380
  > Pretrain Epoch 03 | Avg Loss: 1.5799
  > Pretrain Epoch 04 | Avg Loss: 1.5285
  > Pretrain Epoch 05 | Avg Loss: 1.4749
  > Pretrain Epoch 06 | Avg Loss: 1.4405
  > Pretrain Epoch 07 | Avg Loss: 1.4220
  > Pretrain Epoch 08 | Avg Loss: 1.3801
  > Pretrain Epoch 09 | Avg Loss: 1.3604
  > Pretrain Epoch 10 | Avg Loss: 1.3379

📦 正在切割并锁定 [PATHOLOGICAL] 强非独立同分布数据...
  - Client 1 样本数: 494
  - Client 2 样本数: 480
  - Client 3 样本数: 466

💠 [实验 1/4] 架构: BIDIRECTIONAL (双向) | 防御: NO_DEFENSE
  Round 01/50 | Loss: 1.5782 | Acc: 11.67% ⭐ [Model Saved]
  Round 02/50 | Loss: 1.4283 | Acc: 33.89% ⭐ [Model Saved]
  Round 03/50 | Loss: 1.2955 | Acc: 51.11% ⭐ [Model Saved]
  Round 04/50 | Loss: 1.1718 | Acc: 57.22% ⭐ [Model Saved]
  Round 05/50 | Loss: 1.0625 | Acc: 59.44% ⭐